![image](images/llamaindex.jpeg)


*In response to attendees of NIM workshops who asked about more details on NVIDIA NIM integration with LlamaIndex, we are providing this notebook to demonstrate how to use the NIM API to access the LlamaIndex data.  Credit is to our partners at LlamaIndex (https://docs.llamaindex.ai/en/stable/examples/llm/nvidia/)*
        
        NOTE: Currently this notebook works with API calls but not for running local NIMs.  We are working on a solution for that.  Contact Chris Holland (cholland@nvidia.com) at NVIDIA for more information.

## LlamaIndex is the framework for Context-Augmented LLM Applications
LlamaIndex imposes no restriction on how you use LLMs. You can use LLMs as auto-complete, chatbots, agents, and more. It just makes using them easier. We provide tools like:

- Data connectors ingest your existing data from their native source and format. These could be APIs, PDFs, SQL, and (much) more.
- Data indexes structure your data in intermediate representations that are easy and performant for LLMs to consume.
- Engines provide natural language access to your data. For example:
    - Query engines are powerful interfaces for question-answering (e.g. a RAG flow).
    - Chat engines are conversational interfaces for multi-message, "back and forth" interactions with your data.
- Agents are LLM-powered knowledge workers augmented by tools, from simple helper functions to API integrations and more.
- Observability/Evaluation integrations that enable you to rigorously experiment, evaluate, and monitor your app in a virtuous cycle.
- Workflows allow you to combine all of the above into an event-driven system for flexible than other, graph-based approaches.

# NVIDIA NIMs
The `llama-index-llms-nvidia` package contains LlamaIndex integrations building applications with models on NVIDIA NIM inference microservice. NIM supports models across domains like chat, embedding, and re-ranking models from the community as well as NVIDIA. These models are optimized by NVIDIA to deliver the best performance on NVIDIA accelerated infrastructure and deployed as a NIM, an easy-to-use, prebuilt containers that deploy anywhere using a single command on NVIDIA accelerated infrastructure.

NVIDIA hosted deployments of NIMs are available to test on the NVIDIA API catalog (https://build.nvidia.com/). After testing, NIMs can be exported from NVIDIA’s API catalog using the NVIDIA AI Enterprise license and run on-premises or in the cloud, giving enterprises ownership and full control of their IP and AI application.

NIMs are packaged as container images on a per model basis and are distributed as NGC container images through the NVIDIA NGC Catalog. At their core, NIMs provide easy, consistent, and familiar APIs for running inference on an AI model.


## NVIDIA's LLM connector
This example goes over how to use LlamaIndex to interact with and develop LLM-powered systems using the publicly-accessible AI Foundation endpoints.

With this connector, you'll be able to connect to and generate from compatible models available as hosted NVIDIA NIMs, such as:

- Google's gemma-7b
- Mistal AI's mistral-7b-instruct-v0.2
- And more!

In [ ]:
%pip install --upgrade --quiet llama-index-llms-nvidia llama-index-embeddings-nvidia llama-index-readers-file

### Setup
To get started:

1. Create a free account with NVIDIA, which hosts NVIDIA AI Foundation models.

2. Click on your model of choice.

3. Under Input select the Python tab, and click Get API Key. Then click Generate Key.

4. Copy and save the generated key as NVIDIA_API_KEY. From there, you should have access to the endpoints.

In [ ]:
# Setup Environment
import os
from dotenv import load_dotenv
# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"Current working dir           : { os.getcwd() }")
print(f"NGC_API_KEY for lab use only  : {os.getenv('NGC_API_KEY')}")
print(f"NVIDIA_API_KEY for lab use only  : {os.getenv('NVIDIA_API_KEY')}")

#for use in this notebook only
ngc_api_key = os.getenv('NGC_API_KEY')

In [ ]:
# llama-parse is async-first, running the async code in a notebook requires the use of nest_asyncio
import nest_asyncio

nest_asyncio.apply()

### Working with NVIDIA API Catalog
The NVIDIA API Catalog is a platform that provides access to NVIDIA AI Foundation models. It allows you to test and deploy models on NVIDIA accelerated infrastructure. The NVIDIA API Catalog provides a simple and easy-to-use interface to interact with the models.

In [ ]:
from llama_index.llms.nvidia import NVIDIA
from llama_index.core.llms import ChatMessage, MessageRole

#llm = NVIDIA()
llm = NVIDIA(api_key=ngc_api_key, model="nvdev/mistralai/mistral-7b-instruct-v0.3")
messages = [
    ChatMessage(
        role=MessageRole.SYSTEM, content=("You are a helpful assistant.")
    ),
    ChatMessage(
        role=MessageRole.USER,
        content=("What are the most popular house pets in North America?"),
    ),
]

print(llm.chat(messages))

## Working with NVIDIA NIMs
In addition to connecting to hosted NVIDIA NIMs, this connector can be used to connect to local microservice instances. This helps you take your applications local when necessary.

For instructions on how to setup local microservice instances, see https://developer.nvidia.com/blog/nvidia-nim-offers-optimized-inference-microservices-for-deploying-ai-models-at-scale/

In [ ]:
from llama_index.core import SimpleDirectoryReader, Settings, VectorStoreIndex
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.llms.nvidia import NVIDIA
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

import os

### Loading a specific model
Now we can load our NVIDIA LLM by passing in the model name, as found in the docs - located here

    NOTE: The default model is meta/llama3-8b-instruct. 

In [ ]:
# default model
llm = NVIDIA()
llm.model

We can observe which model our llm object is currently associated with the `.model` attribute.



In [ ]:
llm = NVIDIA(model="nvdev/mistralai/mistral-7b-instruct-v0.3")
llm.model

### Basic Functionality
Now we can explore the different ways you can use the connector within the LlamaIndex ecosystem!

Before we begin, lets set up a list of ChatMessage objects - which is the expected input for some of the methods.

We'll follow the same basic pattern for each example:

1. We'll point our NVIDIA LLM to our desired model
2. We'll examine how to use the endpoint to achieve the desired task!

Complete: `.complete()`
We can use `.complete()`/`.acomplete()` (which takes a string) to prompt a response from the selected model.

Let's use our default model for this task.

In [ ]:
completion_llm = NVIDIA(api_key=ngc_api_key, model="nvdev/mistralai/mistral-7b-instruct-v0.3")


We can verify this is the expected default by checking the `.model` attribute.

In [ ]:
completion_llm.model

Let's call `.complete()` on our model with a string, in this case `"Hello!"`, and observe the response.

In [ ]:
completion_llm.complete("Hello!")


As is expected by LlamaIndex - we get a `CompletionResponse` in response.

*Async Complete*: `.acomplete()`
There is also an async implementation which can be leveraged in the same way!

In [ ]:
await completion_llm.acomplete("Hello!")

**Chat**: `.chat()`
Now we can try the same thing using the `.chat() `method. This method expects a list of chat messages - so we'll use the one we created above.

We'll use the `mistralai/mixtral-8x7b-instruct-v0.1` model for the example.

In [ ]:
chat_llm = NVIDIA(model="nvdev/mistralai/mixtral-8x22b-instruct-v0.1", api_key=ngc_api_key)


All we need to do now is call `.chat()` on our list of `ChatMessages` and observe our response.

You'll also notice that we can pass in a few additional key-word arguments that can influence the generation - in this case, we've used the `seed` parameter to influence our generation and the stop parameter to indicate we want the model to stop generating once it reaches a certain token!

**NOTE**: You can find information about what additional kwargs are supported by the model's endpoint by referencing the API documentation for the selected model. Mixtral's is located here (https://docs.api.nvidia.com/nim/reference/mistralai-mixtral-8x7b-instruct-infer) as an example!

In [ ]:
chat_llm.chat(messages, seed=4, stop=["cat", "cats", "Cat", "Cats"])

As expected, we receive a ChatResponse in response.


**Async Chat**: (`achat`)
We also have an async implementation of the `.chat()` method which can be called in the following way.

In [ ]:
await chat_llm.achat(messages)

**Stream**: `.stream_chat()`
We can also use the models found on *build.nvidia.com* for streaming use-cases!

Let's select another model and observe this behaviour. We'll use Google's` gemma-7b` model for this task.

In [ ]:
stream_llm = NVIDIA(model="nvdev/microsoft/phi-3-medium-4k-instruct", api_key=ngc_api_key)

Let's call our model with `.stream_chat()`, which again expects a list of `ChatMessage` objects, and capture the response.

In [ ]:
streamed_response = stream_llm.stream_chat(messages)

In [ ]:
streamed_response

As we can see, the response is a generator with the streamed response.

Let's take a look at the final response once the generation is complete.

In [ ]:
last_element = None
for last_element in streamed_response:
    pass

print(last_element)

**Async Stream**: `.astream_chat()`
We have the equivalent async method for streaming as well, which can be used in a similar way to the sync implementation.

In [ ]:
streamed_response = await stream_llm.astream_chat(messages)

In [ ]:
streamed_response

In [ ]:
last_element = None
async for last_element in streamed_response:
    pass

print(last_element)

## Tool calling
Starting in v0.2.1, NVIDIA supports tool calling.

NVIDIA provides integration with the variety of models on build.nvidia.com as well as local NIMs. Not all these models are trained for tool calling. Be sure to select a model that does have tool calling for your experimention and applications.

You can get a list of models that are known to support tool calling with,

**NOTE**: For more examples refer : `nvidia_agent.ipynb` (https://docs.llamaindex.ai/en/stable/examples/llm/agent/nvidia_agent.ipynb)

In [ ]:
tool_models = [
    model
    for model in NVIDIA(api_key=ngc_api_key).available_models
    if model.is_function_calling_model
]
tool_models

With a tool capable model,

In [ ]:
# nvdev models dont support tool calling, however you can get your own developer api key and use one of the models above and try.

from llama_index.core.tools import FunctionTool


def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


add_tool = FunctionTool.from_defaults(fn=add)

llm = NVIDIA(api_key=ngc_api_key, model="fill in your model and api key here")
from llama_index.core.agent import FunctionCallingAgentWorker

agent_worker = FunctionCallingAgentWorker.from_tools(
    [multiply_tool, add_tool],
    llm=llm,
    verbose=True,
)
agent = agent_worker.as_agent()

response = agent.chat("What is (121 * 3) + 42?")
print(str(response))

----
### EOF